# Practical 17 — Loops, Goals, and Iterations

The capstone practical is an **iterate-to-target** loop: an agent refines a draft
financial summary until a *deterministic* tool says a measurable goal is met. The
tool `tools/metric.py` scores **coverage** (fraction of required facts present) and
**faithfulness** (no figure invented beyond the source), and `tools/check.py` is the
pass/fail gate that ends the loop. The agent only drafts and reacts to a number it did
not produce — the loop stops *only* when the gate says the target is met.

In the real project you'd drive this from Claude Code / Cline with a `/iterate` skill
and a `PostToolUse` hook. This notebook is the Colab-friendly counterpart: it calls the
same reference tools directly, fully offline, and shows the loop converging on the real
bundled Meridian Robotics earnings release.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "17-loops-goals-iterations"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — The inputs: source text and the goal definition

`data/source.txt` is a fictional Meridian Robotics earnings release. `data/target.json`
defines the goal: a list of **required facts** (each with anchor phrases that must all
appear), a **threshold** for coverage, and the implicit faithfulness rule. We load both
with the tools' own helpers so we use the exact same data the gate sees.

In [ ]:
import json
from tools._common import load_source, load_target

source = load_source()
target = load_target()

print("=== SOURCE (data/source.txt) ===")
print(source)
print("\n=== TARGET (data/target.json) ===")
print("Task     :", target["task"])
print("Threshold:", target["threshold"])
print("Required facts:")
for f in target["facts"]:
    print(f"  - [{f['id']:8s}] {f['label']}")
    print(f"               anchors: {f['must_include']}")

## Step 1 — Score a single candidate

`tools.metric.score(candidate, source, target)` returns the full report: `coverage`,
how many of the `total` facts are `covered`, the list of `missing` facts, whether the
draft is `faithful`, and any `unsupported_figures` (numbers not in the source). Let's
score a deliberately thin one-line draft — it should cover only the revenue fact.

In [ ]:
from tools.metric import score

draft = "Meridian Robotics reported revenue up 22% to 2.4 billion dollars."
report = score(draft, source, target)
print(json.dumps(report, indent=2))

Coverage is 0.2 (1 of 5 facts). The metric *names* exactly which facts are still missing
— that named-gap feedback is what lets the loop add precisely the right fact next.

## Step 2 — The gate decides pass/fail

`tools.check.passes(report)` is the single authority that ends the loop. It returns
`True` only when `coverage >= threshold` **and** `faithful` is `True`. The thin draft is
faithful but well below threshold, so the gate refuses.

In [ ]:
from tools.check import passes

print("Gate passes on the one-line draft?", passes(report))
print(f"  coverage {report['coverage']:.0%}  vs  threshold {report['threshold']:.0%}"
      f"  | faithful={report['faithful']}")

## Step 3 — Run the iterate-to-target loop

Now we simulate the agent. The agent's only job is: read the metric's `missing` list,
add **one** missing fact (pulled faithfully from the source), re-score, and repeat —
stopping the moment `check.passes` is True or after a budget of 5 passes.

We use a small lookup of faithful sentences (one per fact id), each containing only
figures that appear in the source. This stands in for the LLM "drafter": deterministic
here so the notebook is reproducible, but the control flow is identical to the agent's.

In [ ]:
from tools.metric import missing_facts

# A faithful sentence for each required fact — every number occurs in the source.
DRAFT_SENTENCES = {
    "revenue":  "Revenue rose 22% to 2.4 billion dollars.",
    "margin":   "Operating margin expanded to 17% from 13%.",
    "backlog":  "Order backlog reached a record 3.1 billion dollars.",
    "fcf":      "Free cash flow of 310 million dollars.",
    "guidance": "Management issued guidance for fiscal 2026.",
}

MAX_PASSES = 5
facts = target["facts"]

draft = "Meridian Robotics reported fiscal 2025 results."  # deliberately empty of facts
trajectory = []

for step in range(1, MAX_PASSES + 1):
    rep = score(draft, source, target)
    trajectory.append((step, rep["coverage"], rep["faithful"]))
    print(f"pass {step}: coverage {rep['coverage']:.0%} "
          f"({rep['covered']}/{rep['total']})  faithful={rep['faithful']}  "
          f"missing={[m['id'] for m in rep['missing']]}")
    if passes(rep):
        print(f"  -> GATE PASSED at pass {step}. Stopping.")
        break
    # the 'agent' adds exactly the first fact the metric reports as missing
    gap = missing_facts(draft, facts)
    next_fact = gap[0]
    draft += " " + DRAFT_SENTENCES[next_fact["id"]]
    print(f"  -> added missing fact [{next_fact['id']}]")

The loop converges: coverage climbs 0.2 per pass as each named missing fact is added,
and the gate stops the loop the instant `coverage >= 0.8` and `faithful`. The agent
never declared "good enough" — the tool did.

In [ ]:
print("Final draft that met the target:\n")
print(draft)
print("\nScore trajectory (pass, coverage, faithful):")
for step, cov, faith in trajectory:
    bar = "#" * int(round(cov * 20))
    print(f"  pass {step}: {cov:>4.0%} |{bar:<20}| faithful={faith}")

## Step 4 — Faithfulness guard (a "try it" from the README)

The README suggests: *add a number that is NOT in the source and watch faithfulness
flip*. Here we take a draft that covers **all five** facts (coverage 1.0) but sneaks in
a "45% segment margin" — a figure absent from the source. Coverage is full, yet the gate
still refuses, because the agent cannot reach the target by inventing numbers.

In [ ]:
unfaithful = (
    "Revenue rose 22% to 2.4 billion dollars. Operating margin expanded to 17%, "
    "with a one-off segment at 45%. Order backlog reached 3.1 billion dollars. "
    "Free cash flow of 310 million dollars. Guidance for fiscal 2026 was issued."
)
rep_u = score(unfaithful, source, target)
print(json.dumps({
    "coverage": rep_u["coverage"],
    "faithful": rep_u["faithful"],
    "unsupported_figures": rep_u["unsupported_figures"],
    "gate_passes": passes(rep_u),
}, indent=2))
assert rep_u["coverage"] == 1.0 and passes(rep_u) is False
print("\nFull coverage but the gate refuses: the 45% figure is fabricated.")

## Step 5 — Tighten the goal (another "try it"): raise the threshold to 1.0

The README suggests raising `threshold` to 1.0 so every fact becomes mandatory. We don't
edit the bundled file; we just override the loaded target in-memory and rerun the loop.
The four-fact draft that passed at 0.8 now fails, and the loop must take one more pass.

In [ ]:
strict_target = dict(target, threshold=1.0)

draft = "Meridian Robotics reported fiscal 2025 results."
for step in range(1, len(facts) + 2):  # enough budget to cover all five facts
    rep = score(draft, source, strict_target)
    print(f"pass {step}: coverage {rep['coverage']:.0%} "
          f"vs threshold {rep['threshold']:.0%}  faithful={rep['faithful']}")
    if passes(rep):
        print(f"  -> GATE PASSED at pass {step} (every fact present).")
        break
    gap = missing_facts(draft, strict_target["facts"])
    draft += " " + DRAFT_SENTENCES[gap[0]["id"]]

## Recap / next steps

We ran the practical's reference tools end to end, offline:

1. Loaded the source release and the goal definition (`load_source`, `load_target`).
2. Scored candidates with `tools.metric.score` — coverage, missing facts, faithfulness.
3. Gated with `tools.check.passes` — the deterministic authority that ends the loop.
4. Ran the **iterate-to-target loop**, adding the named missing fact each pass and
   watching coverage climb 0.2 at a time until the gate passed.
5. Confirmed the faithfulness guard rejects a fabricated figure even at full coverage.
6. Tightened the threshold to 1.0 and saw the loop take an extra pass.

Next steps to explore: add a sixth required fact to `data/target.json` and watch the
loop run one more pass; shrink `MAX_PASSES` to 2 and confirm the loop stops *below*
target and reports the best score instead of pretending success; then run the full
agentic version in Claude Code / Cline with `/iterate`, where a `PostToolUse` hook
re-scores the draft automatically after every edit. Run `python -m pytest -q` to verify
the loop's invariants.